## Prepare Env

In [ ]:
! pip install 'acryl-datahub[delta-lake]'

In [ ]:
import os

In [ ]:
# Define S3 storage
obj_storage_access_key = os.getenv('OBJ_STORAGE_ACCESS_KEY', 'demo-access-key')
obj_storage_secret_key = os.getenv('OBJ_STORAGE_SECRET_KEY', 'demo-secret-key')
obj_storage_endpoint = os.getenv('OBJ_STORAGE_ENDPOINT', 'http://192.168.1.79:9000')

### 1. Add delta table

In [ ]:
yaml_content = f"""
source:
  type: "delta-lake"
  config:
    base_path: "s3://warehouse/bronze/opensanctions_entities.delta"
    s3:
      aws_config:
        aws_access_key_id: "{obj_storage_access_key}"
        aws_secret_access_key: "{obj_storage_secret_key}"
        aws_endpoint_url: "{obj_storage_endpoint}"
        aws_region: us-west-2

sink:
  type: "datahub-rest"
  config:
    server: "http://localhost:8080"
"""

with open('delta.s3.dhub.yaml', 'w') as file:
    file.write(yaml_content)


In [ ]:
! datahub ingest -c delta.s3.dhub.yaml

### 2. Add S3 data lake files

In [ ]:
yaml_content = f"""
source:
  type: s3
  config:
    path_specs:
      - include: "s3://warehouse/files/{{datasource}}/*.*"

    aws_config:
      aws_access_key_id: "{obj_storage_access_key}"
      aws_secret_access_key: "{obj_storage_secret_key}"
      aws_endpoint_url: "{obj_storage_endpoint}"
      aws_region: us-east-2
    env: "TEST"
    profiling:
      enabled: false
sink:
  type: "datahub-rest"
  config:
    server: "http://localhost:8080"
"""
with open('layer-files.s3.dhub.yaml', 'w') as file:
    file.write(yaml_content)

In [ ]:
! datahub ingest -c layer-files.s3.dhub.yaml